# YOLOv5 object detection: train, test, export to the Android app
Mask / no mask project from the course session (Roboflow). Run all cells on a Colab GPU (Runtime > Change runtime type > T4).
The last cell downloads `detector.tflite` and `detector_classes.txt`. Put both in `app/src/main/assets/models/` and rebuild the app.

In [ ]:
#clone YOLOv5 and install dependencies
!git clone https://github.com/ultralytics/yolov5  # clone repo
%cd yolov5
%pip install -qr requirements.txt
%pip install -q roboflow

import torch, os
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

## Dataset
The API key is NOT stored in this file. In Colab open the key icon (Secrets), add `ROBOFLOW_API_KEY`, and turn on notebook access.
If an old copy of this notebook with the key inside was shared anywhere, regenerate the key in Roboflow settings.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("cvproject-ctn09").project("cv_project-htqly")
dataset = project.version(4).download("yolov5")
print(dataset.location)

In [ ]:
import yaml
with open(f"{dataset.location}/data.yaml") as f:
    names = yaml.safe_load(f)["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
print(len(names), names)

## Train (img 480, 50 epochs; the earlier run reached mAP50 0.84)

In [ ]:
!python train.py --img 480 --batch 16 --epochs 50 --data {dataset.location}/data.yaml --weights yolov5s.pt --cache --name cvproj

In [ ]:
# Optional: TensorBoard curves
%load_ext tensorboard
%tensorboard --logdir runs

## Test on the test images

In [ ]:
!python detect.py --weights runs/train/cvproj/weights/best.pt --img 480 --conf 0.25 --source {dataset.location}/test/images --name cvproj_test
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('runs/detect/cvproj_test/*.jpg'))[:6]:
    display(Image(p))

## Export to TFLite for the app
Uses the same 480 input size as training.

In [ ]:
!python export.py --weights runs/train/cvproj/weights/best.pt --include tflite --img 480
import glob
tfl = sorted(glob.glob('runs/train/cvproj/weights/*.tflite'))
print(tfl)

In [ ]:
# Pick the float32 file (the app feeds float input) and write the class list next to it.
import shutil, glob
src = [p for p in glob.glob('runs/train/cvproj/weights/*.tflite') if 'int8' not in p][0]
shutil.copy(src, '/content/detector.tflite')
open('/content/detector_classes.txt', 'w').write('\n'.join(names) + '\n')

# Sanity check: the output must be [1, N, 5 + number of classes]
import tensorflow as tf
it = tf.lite.Interpreter(model_path='/content/detector.tflite')
print('input ', it.get_input_details()[0]['shape'])
print('output', it.get_output_details()[0]['shape'], '=> classes expected:', len(names))

In [ ]:
from google.colab import files
files.download('/content/detector.tflite')
files.download('/content/detector_classes.txt')
# Also keep the PyTorch weights for later
files.download('runs/train/cvproj/weights/best.pt')